<a href="https://colab.research.google.com/github/AyseNil-star/AyseNil-star/blob/main/week09_humanisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9 — Humanisation and Immunogenicity

**Computational Antibody Engineering**

Week 8 asked whether a molecule can be *manufactured*. Today asks whether it can be *given to a person*
more than once.

Most antibodies begin life in a mouse. A mouse antibody injected into a human is a foreign protein, and
the patient's immune system responds by making **anti-drug antibodies** (ADAs) against it. ADAs
neutralise the drug, accelerate its clearance, and occasionally cause serious reactions. A murine
antibody is a superb reagent and, usually, an unusable medicine.

**Humanisation** is the process of making the molecule look human to the immune system while keeping it
bound to its target. Those two goals pull against each other, and today you measure the tension.

> **How human can you make a murine antibody before you break it?**

---
### Today's antibody
The murine Fab **`4G6K`** — the one that came out with three high-risk liabilities in week 8's panel.
We now try to turn it into something you could put in a person.

## 0. Setup

**Sapiens** is the humanising model behind BioPhi. It is a transformer trained on human antibody
repertoires: give it an antibody sequence and it rewrites residues toward what a human repertoire would
have produced.

We also need ANARCI (numbering and germline assignment) and ABodyBuilder2 (structure), both from
earlier weeks.

In [1]:
%pip install -q sapiens anarci ImmuneBuilder openmm pdbfixer biopython pandas matplotlib
!apt-get install -qq hmmer > /dev/null 2>&1

import warnings
warnings.filterwarnings("ignore")

import os, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sapiens
from anarci import anarci, number
from ImmuneBuilder import ABodyBuilder2
from Bio.PDB import PDBParser, Superimposer
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.Polypeptide import three_to_index, index_to_one

parser = PDBParser(QUIET=True)
sasa_calc = ShrakeRupley()
STANDARD = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE",
            "LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"}
MAX_ASA = {"ALA":129,"ARG":274,"ASN":195,"ASP":193,"CYS":167,"GLN":225,"GLU":223,
           "GLY":104,"HIS":224,"ILE":197,"LEU":201,"LYS":236,"MET":224,"PHE":240,
           "PRO":159,"SER":155,"THR":172,"TRP":285,"TYR":263,"VAL":174}
IMGT_CDR = {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)}
print("Ready.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Ready.


## 1. Why humanise, and what "human" means

An antibody's **framework** is a conserved scaffold; its **CDRs** are the variable loops that bind. A
mouse framework differs from a human one at many positions, and those differences are what a human
immune system sees as foreign.

The historical progression of therapeutic antibodies is a progression in how much mouse is left:

| Generation | What it is | Suffix | Immunogenicity |
|---|---|---|---|
| murine | entirely mouse | -omab | high |
| chimeric | mouse variable domains, human constant | -ximab | reduced |
| **humanised** | mouse CDRs on a human framework | -zumab | low |
| fully human | human throughout | -umab | lowest |

**Humanness** is measured as similarity to human antibody sequences. We will use the simplest defensible
metric: **identity to the closest human germline V gene**, which ANARCI computes directly (week 2).

That is a proxy, and worth being honest about. Real immunogenicity depends on which peptides are
presented by a patient's MHC, and MHC alleles vary between people. A humanness score is a *risk
indicator*, not a prediction of what a given patient's immune system will do.

In [2]:
def download(pdb_id):
    path = f"{pdb_id}.pdb"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
    return parser.get_structure(pdb_id, path)[0]


def chain_sequence(model, chain_id):
    return "".join(index_to_one(three_to_index(r.get_resname()))
                   if r.get_resname() in STANDARD else "X"
                   for r in model[chain_id] if r.id[0] == " ")


def variable_domain(seq):
    numbering, _ = number(seq, scheme="imgt")
    occupied = [(p, a) for (p, ic), a in numbering if a != "-"]
    return "".join(a for p, a in occupied), [p for p, a in occupied]


def in_cdr(imgt):
    return any(lo <= imgt <= hi for lo, hi in IMGT_CDR.values())


def region_of(imgt):
    for name, (lo, hi) in IMGT_CDR.items():
        if lo <= imgt <= hi:
            return name
    return "framework"


m = download("4G6K")
VH, POS_H = variable_domain(chain_sequence(m, "H"))
VL, POS_L = variable_domain(chain_sequence(m, "L"))

parent = {"H": VH, "L": VL}
positions = {"H": POS_H, "L": POS_L}

for ch in ("H", "L"):
    cdrs = {region_of(p): "" for p in positions[ch] if in_cdr(p)}
    for i, p in enumerate(positions[ch]):
        if in_cdr(p):
            cdrs[region_of(p)] += parent[ch][i]
    print(f"murine V{ch} ({len(parent[ch])} aa)")
    for name in ["CDR1", "CDR2", "CDR3"]:
        print(f"   {ch}-{name}: {cdrs.get(name)}")

murine VH (120 aa)
   H-CDR1: GFSLSTSGMG
   H-CDR2: IWWDGDE
   H-CDR3: ARNRYDPPWFVD
murine VL (107 aa)
   L-CDR1: QDISNY
   L-CDR2: YTS
   L-CDR3: LQGKMLPWT


In [3]:
def humanness(seq):
    """Identity to the closest HUMAN germline V gene. Higher = more human-looking."""
    _, details, _ = anarci([("x", seq)], scheme="imgt",
                           assign_germline=True, allowed_species=["human"])
    gene, identity = details[0][0]["germlines"]["v_gene"]
    return gene[1], identity


print(f"{'chain':<8}{'closest human V gene':<24}{'identity'}")
print("-" * 46)
baseline = {}
for ch in ("H", "L"):
    gene, ident = humanness(parent[ch])
    baseline[ch] = ident
    print(f"{ch:<8}{gene:<24}{ident:.3f}")

print("\nFor comparison, the approved antibodies from week 8 sat around 0.78-0.87 human identity.")

chain   closest human V gene    identity
----------------------------------------------
H       IGHV4-38-2*01           0.755
L       IGKV1-39*01             0.837

For comparison, the approved antibodies from week 8 sat around 0.78-0.87 human identity.


## 2. Hand it to the model

Sapiens takes an antibody sequence and returns a humanised one of the same length. Let's simply run it
and see what it does.

In [4]:
naive = {ch: sapiens.predict_best_score(parent[ch], chain_type=ch) for ch in ("H", "L")}

rows = []
for ch in ("H", "L"):
    for i, p in enumerate(positions[ch]):
        if parent[ch][i] != naive[ch][i]:
            rows.append({"chain": ch, "imgt": p, "from": parent[ch][i],
                         "to": naive[ch][i], "region": region_of(p)})

changes = pd.DataFrame(rows)
for ch in ("H", "L"):
    sub = changes[changes.chain == ch]
    n_cdr = (sub.region != "framework").sum()
    print(f"chain {ch}: {len(sub)}/{len(parent[ch])} residues changed "
          f"({100*len(sub)/len(parent[ch]):.0f}%)  -  {n_cdr} of them in CDRs")

print("\nWhat happened to the CDRs:")
for ch in ("H", "L"):
    for name in ["CDR1", "CDR2", "CDR3"]:
        before = "".join(parent[ch][i] for i, p in enumerate(positions[ch])
                         if region_of(p) == name)
        after = "".join(naive[ch][i] for i, p in enumerate(positions[ch])
                        if region_of(p) == name)
        flag = "   <-- CHANGED" if before != after else ""
        print(f"   {ch}-{name}:  {before:<28} -> {after}{flag}")

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.28MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.28MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

chain H: 11/120 residues changed (9%)  -  6 of them in CDRs
chain L: 15/107 residues changed (14%)  -  7 of them in CDRs

What happened to the CDRs:
   H-CDR1:  GFSLSTSGMG                   -> GGSLSTSGMY   <-- CHANGED
   H-CDR2:  IWWDGDE                      -> IYWDGDT   <-- CHANGED
   H-CDR3:  ARNRYDPPWFVD                 -> ARNRYDPPWFDP   <-- CHANGED
   L-CDR1:  QDISNY                       -> QDISNY
   L-CDR2:  YTS                          -> AAS   <-- CHANGED
   L-CDR3:  LQGKMLPWT                    -> QQYKSYPYT   <-- CHANGED


### Stop. Look at what it did to the CDRs.

Sapiens rewrote the binding loops. Light chain CDR3 went from one sequence to a completely different
one — not a conservative substitution or two, a different loop.

The model did exactly what it was asked. It was trained to make sequences look like human repertoire
sequences, and murine CDRs do not look like human CDRs, so it changed them. Humanness went up.

**And the antibody almost certainly no longer binds anything.** CDR-L3 was a major contributor to the
paratope in every antibody we have examined. You cannot replace it and expect the molecule to keep its
target.

This is the central trap of automated humanisation, and it generalises well beyond this tool: **an
optimiser will happily optimise away the thing you forgot to tell it to protect.** The objective was
humanness. Binding was never in the objective, so binding was spent.

## 3. CDR grafting — protect what does the work

The classical answer is **CDR grafting**: keep the humanised framework, put the original CDRs back.

The logic is that the framework is a scaffold and the CDRs are the functional part, so you can swap
scaffolds and carry the function across. We implement it in four lines.

In [5]:
grafted = {}
for ch in ("H", "L"):
    seq = list(naive[ch])
    for i, p in enumerate(positions[ch]):
        if in_cdr(p):
            seq[i] = parent[ch][i]          # restore the murine CDR residue
    grafted[ch] = "".join(seq)

summary = []
for ch in ("H", "L"):
    n_cdr = sum(1 for i, p in enumerate(positions[ch])
                if in_cdr(p) and grafted[ch][i] != parent[ch][i])
    n_fw = sum(1 for i, p in enumerate(positions[ch])
               if not in_cdr(p) and grafted[ch][i] != parent[ch][i])
    summary.append({"chain": ch, "CDR changes": n_cdr, "framework changes": n_fw})
print(pd.DataFrame(summary).to_string(index=False))

variants = {"murine parent": parent, "Sapiens (naive)": naive, "CDR-grafted": grafted}

rows = []
for name, v in variants.items():
    row = {"variant": name}
    for ch in ("H", "L"):
        row[f"humanness_{ch}"] = round(humanness(v[ch])[1], 3)
    rows.append(row)

humanness_table = pd.DataFrame(rows)
humanness_table

chain  CDR changes  framework changes
    H            0                  5
    L            0                  8


,variant,humanness_H,humanness_L
0,murine parent,0.755,0.837
1,Sapiens (naive),0.837,0.946
2,CDR-grafted,0.806,0.913


### YOUR TURN — Question 1

1. By how much did naive humanisation raise human identity, for each chain?
2. Restoring the CDRs gives **some of that gain back**. How much, for each chain? Why does putting
   murine residues back lower the humanness score?
3. The CDR-grafted variant is less human than the naive one but more human than the parent. Is that a
   good trade? What would you need to know to answer properly?
4. Look at the week 8 panel figures quoted in section 1. Is the grafted variant now in the range of
   approved antibodies?

*Double-click and write your answers here.*

1.
2.
3.
4.

**That is the humanisation trade, quantified.** Every murine residue you keep for binding costs you
humanness; every framework residue you humanise is a residue the immune system will not flag. The
grafted variant sits between the two extremes because that is exactly what it is — a compromise.

What the score cannot tell you is whether the molecule still binds. For that we need structure.

## 4. Does the graft actually preserve the binding site?

The premise of CDR grafting is that CDR **sequence** carries the binding function. But binding depends
on CDR **conformation** — the three-dimensional shape of those loops — and the loops are held in place
by the framework underneath them.

If you change the framework, you may change the shape of loops whose sequence you carefully preserved.

Let's test it: predict all three variants and compare CDR backbone positions after superposing on the
framework (week 4's method).

In [6]:
predictor = ABodyBuilder2()

for name, v in variants.items():
    path = f"wk9_{name.split()[0].lower().strip('()')}.pdb"
    if not os.path.exists(path):
        predictor.predict({"H": v["H"], "L": v["L"]}).save(path)
    variants[name]["_file"] = path
    print(f"  predicted {name}")

models = {name: parser.get_structure(name, v["_file"])[0]
          for name, v in variants.items()}


def cdr_shift(reference, variant):
    """RMSD by region after superposing two predicted Fvs on their frameworks."""
    pairs = []
    for ch in ("H", "L"):
        a = {r.id[1]: r for r in models[reference][ch] if r.id[0] == " "}
        b = {r.id[1]: r for r in models[variant][ch] if r.id[0] == " "}
        for p in sorted(set(a) & set(b)):
            if "CA" in a[p] and "CA" in b[p]:
                label = "framework" if not in_cdr(p) else f"{ch}-{region_of(p)}"
                pairs.append((label, a[p]["CA"], b[p]["CA"]))

    fw = [(x, y) for lab, x, y in pairs if lab == "framework"]
    sup = Superimposer()
    sup.set_atoms([x for x, y in fw], [y for x, y in fw])
    sup.apply([y for lab, x, y in pairs])

    grouped = {}
    for lab, x, y in pairs:
        grouped.setdefault(lab, []).append((x - y) ** 2)
    return {lab: round(float(np.sqrt(np.mean(v))), 2) for lab, v in grouped.items()}


shift = pd.DataFrame({
    "CDR-grafted vs parent": cdr_shift("murine parent", "CDR-grafted"),
    "naive vs parent": cdr_shift("murine parent", "Sapiens (naive)"),
}).reindex(["framework", "H-CDR1", "H-CDR2", "H-CDR3",
            "L-CDR1", "L-CDR2", "L-CDR3"])
shift

  predicted murine parent
  predicted Sapiens (naive)
  predicted CDR-grafted


,CDR-grafted vs parent,naive vs parent
framework,0.50,0.60
H-CDR1,0.60,1.21
H-CDR2,0.93,1.38
H-CDR3,0.46,1.03
L-CDR1,0.40,0.67
L-CDR2,0.28,0.28
L-CDR3,0.23,0.60


### The graft did not preserve the CDRs

Read the first column. Those CDR sequences are **identical** to the parent's — we put them back
ourselves. And several of them have still moved, one of them by roughly twice the framework RMSD.

The loops are anchored on a framework we changed, so they sit differently.

This is the **vernier zone** effect. A subset of framework residues, packed directly underneath the
CDRs, act like the vernier on a caliper: small changes there produce fine adjustments in loop
positioning. CDR grafting that ignores them frequently produces a humanised antibody with badly reduced
affinity — historically this was *the* problem with the technique, and the fix is to "back-mutate"
selected framework positions to the murine residue, trading humanness for binding.

Which raises the practical question: **which framework positions are the risky ones?**

In [7]:
ref = models["murine parent"]

rows = []
for ch in ("H", "L"):
    residues = {r.id[1]: r for r in ref[ch] if r.id[0] == " "}
    cdr_atoms = [a for p, r in residues.items() if in_cdr(p) for a in r]
    for i, p in enumerate(positions[ch]):
        if not in_cdr(p) and grafted[ch][i] != parent[ch][i]:
            r = residues.get(p)
            d = min(np.linalg.norm(a.coord - b.coord) for a in r for b in cdr_atoms)
            rows.append({"chain": ch, "imgt": p, "murine": parent[ch][i],
                         "human": grafted[ch][i], "dist_to_CDR": round(d, 1),
                         "vernier": d < 5.0})

vernier = pd.DataFrame(rows).sort_values("dist_to_CDR")
print(f"{len(vernier)} framework changes; "
      f"{vernier.vernier.sum()} lie within 5 A of a CDR residue\n")
print(vernier.to_string(index=False))

13 framework changes; 6 lie within 5 A of a CDR residue

chain  imgt murine human  dist_to_CDR  vernier
    H    80      K     V     1.800000     True
    L    40      S     N     1.900000     True
    H    25      F     V     2.200000     True
    L    87      Y     F     2.200000     True
    H    87      V     F     2.300000     True
    L   103      F     Y     4.200000     True
    L    68      H     Q     6.500000    False
    L    50      V     P     6.800000    False
    H    91      I     L    11.100000    False
    L     8      T     P    12.600000    False
    L   124      L     V    14.400000    False
    H    46      S     P    16.299999    False
    L    96      Q     P    20.799999    False


### You have just identified vernier residues from structure

No lookup table, no published list — a distance calculation on a predicted structure.

The ones within about 5 Å pack against the CDR loops. Those are the framework positions where
humanisation is most likely to cost you binding, and therefore the first candidates for back-mutation
if the humanised antibody loses affinity.

The ones far from the CDRs are, structurally speaking, free humanisation: they change how human the
molecule looks without touching the machinery that binds.

**That is a genuinely useful output.** It turns "humanise and hope" into a ranked list: humanise
everything far from the CDRs, and treat the close ones as a small set of explicit, testable decisions.

## 5. Did humanisation change the developability picture?

Week 8's screen, re-run on all three variants. Humanisation changes sequence, so it can create or
destroy liability motifs.

In [8]:
import re

MOTIFS = [("deamidation","NG",r"NG"), ("deamidation","NS",r"NS"),
          ("isomerisation","DG",r"DG"), ("isomerisation","DS",r"DS"),
          ("fragmentation","DP",r"DP"), ("oxidation","M",r"M"),
          ("glycosylation","NxS/T",r"N[^P][ST]")]


def high_risk_liabilities(name):
    """Week 8's screen: motifs that are both exposed and in a CDR."""
    model = parser.get_structure("x", variants[name]["_file"])[0]
    for c in list(model):
        for r in [r for r in model[c.id] if r.id[0] != " "]:
            model[c.id].detach_child(r.id)
    sasa_calc.compute(model, level="R")

    found = []
    for ch in ("H", "L"):
        seq = variants[name][ch]
        residues = [r for r in model[ch] if r.id[0] == " "]
        rsa = {j: residues[j].sasa / MAX_ASA.get(residues[j].get_resname(), 200)
               for j in range(len(residues))}
        for kind, label, pattern in MOTIFS:
            for mt in re.finditer(pattern, seq):
                j = mt.start()
                p = positions[ch][j]
                if in_cdr(p) and rsa[j] > 0.25:
                    found.append(f"{ch}{p} {label}")
    return found


for name in variants:
    hits = high_risk_liabilities(name)
    print(f"{name:<18}{len(hits)} high-risk liabilities   {hits}")

murine parent     3 high-risk liabilities   ['H59 DG', 'H110 DP', 'L109 M']
Sapiens (naive)   1 high-risk liabilities   ['H59 DG']
CDR-grafted       3 high-risk liabilities   ['H59 DG', 'H110 DP', 'L109 M']


### A result that should make you uneasy

Naive humanisation **removed liabilities**. It looks better on week 8's screen than the molecule we
actually want to develop.

It achieved that by rewriting the CDRs — the same act that destroyed the binding site. On a dashboard
showing humanness and developability, the naive variant wins on both axes. It is also useless.

The CDR-grafted variant keeps every one of the parent's liabilities, because liabilities live in the
CDRs and we deliberately preserved them.

**The lesson is about optimisation, not about antibodies.** Every metric in this course measures
something real and partial. Optimise any one of them hard enough and you will find a molecule that
scores brilliantly and does not work. The binding assay is not one of the metrics on the dashboard —
which is exactly why it has to stay in the loop.

## 6. Deliverable

**Submit:** this notebook executed top to bottom, committed as `week09/week09_humanisation.ipynb`.

Must contain: the naive humanisation changes, the humanness table, the CDR shift table, the vernier
analysis, the liability comparison, your answers to Question 1, and the report below.

In [9]:
print("=" * 68)
print("WEEK 9 SUMMARY - humanising murine 4G6K")
print("=" * 68)
print(f"{'variant':<18}{'human ident H':<15}{'human ident L':<15}{'liabilities'}")
for name in variants:
    h = humanness(variants[name]["H"])[1]
    l = humanness(variants[name]["L"])[1]
    print(f"{name:<18}{h:<15.3f}{l:<15.3f}{len(high_risk_liabilities(name))}")
print()
print("CDR movement caused by framework humanisation alone (identical CDR sequences):")
for region in ["H-CDR1", "H-CDR2", "H-CDR3", "L-CDR1", "L-CDR2", "L-CDR3"]:
    print(f"   {region:<10}{shift.loc[region, 'CDR-grafted vs parent']:.2f} A")
print(f"   {'framework':<10}{shift.loc['framework', 'CDR-grafted vs parent']:.2f} A")
print()
print(f"Framework changes within 5 A of a CDR (vernier candidates): "
      f"{int(vernier.vernier.sum())} of {len(vernier)}")
print("=" * 68)

WEEK 9 SUMMARY - humanising murine 4G6K
variant           human ident H  human ident L  liabilities
murine parent     0.755          0.837          3
Sapiens (naive)   0.837          0.946          1
CDR-grafted       0.806          0.913          3

CDR movement caused by framework humanisation alone (identical CDR sequences):
   H-CDR1    0.60 A
   H-CDR2    0.93 A
   H-CDR3    0.46 A
   L-CDR1    0.40 A
   L-CDR2    0.28 A
   L-CDR3    0.23 A
   framework 0.50 A

Framework changes within 5 A of a CDR (vernier candidates): 6 of 13


### Your report

*Double-click and write 250–350 words.*

You have humanised a murine antibody. Write the memo that goes with the sequences you are handing to the
protein engineers.

Cover:

- what you did, and why you did not simply use the model's output directly
- the humanness you achieved, per chain, and what it cost relative to the naive version
- the **specific framework positions** you would flag for back-mutation if the humanised antibody loses
  affinity, and why those ones
- what the structural comparison does and does not establish
- what you want measured before anyone proceeds

**One thing you must handle honestly.** The CDR shifts you measured are differences between two
*predicted* structures. Week 4 measured ABodyBuilder2's own error against crystal structures — go back
and look at that number. Is the shift you are reporting comfortably larger than the predictor's error,
or is it the same size? Say so explicitly, and let it set how strongly you word the memo.

---

**Your answer:**

## 7. Take-home exercises

Due before week 10.

**Exercise 1 — back-mutation (required).**
Pick the two vernier positions closest to a CDR and revert them to the murine residue. Recompute
humanness and re-run the structural comparison. Did the CDR conformations move back toward the parent?
What did it cost in humanness? This is the actual engineering loop, run once.

**Exercise 2 — how much is predictor noise? (required).**
Predict the *parent* twice and compare the two predictions to each other. ABodyBuilder2 is
deterministic, so instead compare the parent against a variant with a **single** conservative framework
mutation far from any CDR (use your vernier table to pick one). How large a CDR shift does a change
that *should* be harmless produce? Use that as your noise floor, and re-judge the shifts in section 4
against it.

**Exercise 3 — the real tools (stretch).**
Run the parent through [BioPhi](https://biophi.dichlab.org/) (Sapiens humanisation plus **OASis**
humanness scoring) and [Hu-mAb](https://opig.stats.ox.ac.uk/webapps/humab/). OASis scores humanness by
peptide-level prevalence in human repertoires — a different and more clinically motivated definition
than germline identity. Do the tools agree on which positions to humanise? Where they disagree, what
would you do? That disagreement is the normal condition, not an anomaly.

In [10]:
# Exercise 1 - back-mutation.
# `vernier` holds the candidate positions; `grafted` holds the current sequences.

# YOUR CODE HERE

In [11]:
# Exercise 2 - establish a noise floor for the CDR shift measurement.

# YOUR CODE HERE

---

### Where we go next

Today you used a language model (Sapiens) as a black box that rewrites sequences. Next week you open
that box: what these models learn from sequence alone, what their scores mean, and how to use them to
rank mutations rather than to make them for you.

| Week | Relationship to today |
|---|---|
| **10** | Protein language models — Sapiens is one; next week you use them deliberately |
| **11** | Inverse folding: redesign guided by structure, intersected with sequence-model scores |
| **12** | Designing CDRs from nothing, where "the optimiser spent the thing you forgot to protect" is the permanent risk |
| **13** | Assembling these filters, and how a multi-objective dashboard misleads |

Before week 10: finish exercises 1 and 2, and commit.

**One habit from today.** When you hand a molecule to an optimiser, write down what must not change
before you start. The model will not infer it, and it will spend anything you leave unprotected.